<div style="text-align: center; padding: 25px; background: linear-gradient(135deg, #37474f 0%, #455a64 100%); color: white; border-radius: 10px; margin-bottom: 15px;">

<h1 style="color: white; margin: 0; font-size: 28px; font-weight: bold;">Adquisición Automatizada de Imágenes Satelitales (Google Maps)</h1>

<h3 style="color: #cfd8dc; margin-top: 10px; font-weight: bold;">Metodología · Fase 1: Construcción del Banco de Imágenes</h3>

<p style="color: #eceff1; margin-top: 18px; margin-bottom: 0; font-size: 14px; font-style: italic; font-weight: bold;">Trabajo de grado sobre clasificación sistemática de depresiones sub-circulares (SCD-H₂) mediante YOLOv8 en la Alta Guajira</p>

</div>

<div style="padding: 15px; background-color: #eceff1; border-left: 5px solid #37474f; margin-bottom: 20px; color: #212121;">

<b>Autor:</b> Miguel Ángel Monterroza Montes<br>
<b>Institución:</b> Universidad del Norte — Barranquilla, Colombia<br>
<b>Programa:</b> Geología<br>
<b>Periodo académico:</b> 2026-30

</div>

---


### **Propósito de este notebook**

Este notebook implementa el pipeline de adquisición del banco de imágenes satelitales de entrenamiento que constituye la Fase 1 de la metodología del trabajo de grado. Automatiza la descarga de 600 imágenes satelitales del servicio **Google Maps Static API** (60 estructuras por cada una de las 10 clases del inventario), con parámetros calibrados empíricamente para maximizar la calidad y la homogeneidad del contenido visual disponible para el entrenamiento posterior del modelo (clasificación supervisada de YOLOv8). Este mismo pipeline se reutiliza, con ajustes menores documentados más adelante en este mismo encabezado, para adquirir el banco de imágenes de aplicación (inventario preexistente de estructuras identificadas en la Alta Guajira mediante fotointerpretación visual) sobre las que el modelo entrenado emite una clasificación para la posterior construcción de un inventario final de SCD-H₂ candidatas a validación geoquímica de campo en la región.

Cada imagen se obtiene mediante una **consulta HTTP** individual al servicio: el pipeline construye una dirección web (URL) con los parámetros de localización, escala y estilo específicos de cada estructura, la envía al servidor de Google, y recibe como respuesta el archivo PNG correspondiente. Este mecanismo reemplaza la automatización de capturas de pantalla directamente en la web costumer de Google Maps —empleada en Ginzburg et al. (2025)— por una solicitud directa y estandarizada al servidor cartográfico, lo que permite obtener imágenes con resolución nativa superior, sin elementos de interfaz gráfica superpuestos y de manera plenamente reproducible en cualquier entorno.

### **Alcance funcional**

El pipeline produce, para cada estructura del CSV de entrada, una única versión recortada de la imagen satelital descargada a resolución nativa 1280 × 1280 píxeles. El recorte es uniforme de 1184 × 1184 píxeles para todas las clases del inventario, lo que elimina cualquier factor de escala diferencial entre clases durante el entrenamiento posterior del modelo y garantiza homogeneidad de escala entre las imágenes que alimentan el entrenamiento y aquellas sobre las cuales el modelo emitirá su clasificación en la fase de aplicación. Este tamaño es múltiplo de 32, condición nativa de las arquitecturas convolucionales utilizadas por YOLOv8.

### **Reutilización del pipeline para otros CSV de entrada**

El mismo notebook se reutiliza sin modificar su estructura funcional para adquirir imágenes a partir de cualquier CSV que respete el formato del maestro (columnas `id`, `clase`, `X_3857`, `Y_3857`, `zoom`; separador `;` y decimal `,`; coordenadas en EPSG:3857). Basta con ajustar tres valores del diccionario `CONFIG` en la sección 3: `csv_entrada` para apuntar al nuevo CSV, `carpeta_dataset` para dirigir las imágenes descargadas a una carpeta de salida propia, y `log_csv` para asignar un archivo de trazabilidad independiente. En el caso concreto del banco de imágenes de aplicación, por ejemplo, esos tres valores se sustituyen por los correspondientes al inventario de estructuras candidatas identificadas mediante fotointerpretación en la Alta Guajira, sin necesidad de modificar ninguna función auxiliar. La ausencia de contaminación entre ejecuciones queda garantizada por el aislamiento físico de artefactos: cada CSV escribe en su propia carpeta y su propio log, y la lógica de reanudación consulta únicamente el log activo.

<h2 style="font-weight: bold; border-bottom: 3px solid #37474f; padding-bottom: 8px;">1. Importaciones</h2>

El pipeline utiliza siete librerías del ecosistema Python. Cinco de ellas son de la biblioteca estándar y se instalan automáticamente con la distribución; las restantes se instalan a través de los gestores `pip` o `conda`:

- **`pandas`** — lectura y manipulación tabular del CSV maestro. Permite recorrer las 600 filas del dataset de manera eficiente.
- **`pyproj`** — transformación entre sistemas de referencia de coordenadas. El CSV almacena coordenadas en proyección Web Mercator (EPSG:3857), mientras que la Static API acepta coordenadas geográficas WGS 84 (EPSG:4326).
- **`requests`** — envío de peticiones HTTP a la Static API. Alternativa moderna al módulo `urllib` de la biblioteca estándar, con manejo más limpio de sesiones, timeouts y códigos de respuesta.
- **`Pillow` (PIL)** — procesamiento de imágenes. Se usa para cargar el contenido binario descargado como un objeto de imagen y aplicar los recortes geométricos.

Los módulos `os`, `sys`, `time`, `csv`, `datetime`, `pathlib` e `io` provienen de la biblioteca estándar y no requieren instalación adicional.


In [ ]:
import os
import sys
import time
import csv
from datetime import datetime
from pathlib import Path
from io import BytesIO

import pandas as pd
from pyproj import Transformer
import requests
from PIL import Image

<h2 style="font-weight: bold; border-bottom: 3px solid #37474f; padding-bottom: 8px;">2. Configuración de la API Key</h2>

La Google Maps Static API requiere una clave de autenticación (API key) para procesar cada petición. Esta clave identifica al proyecto de Google Cloud responsable del consumo y permite a Google contabilizarlo contra la cuota mensual gratuita.

La clave debe reemplazar la cadena `TU_API_KEY_AQUI` antes de la ejecución. El notebook está diseñado de forma que la clave nunca aparezca embebida en el código publicado: el placeholder actúa como marcador visible y una validación al inicio del flujo detiene la ejecución si no se ha reemplazado, evitando peticiones sin credenciales que resultarían en respuestas de error.

Los detalles sobre cómo obtener esta clave, restringir su uso y monitorear el consumo se presentan en el manuscrito del trabajo de grado.

In [ ]:
API_KEY = "TU_API_KEY_AQUI" # Reemplaza con tu propia API Key de Google Maps Static API

<h2 style="font-weight: bold; border-bottom: 3px solid #37474f; padding-bottom: 8px;">3. Configuración centralizada del pipeline</h2>

Todos los parámetros operativos del pipeline se concentran en un único diccionario. Esta centralización facilita cualquier ajuste posterior (nombres de archivo, tiempos de espera, políticas de reintento) sin necesidad de modificar el cuerpo de las funciones. El diccionario se agrupa temáticamente:

- **Rutas de entrada/salida:** Archivo CSV de entrada, carpeta con imágenes y log de trazabilidad (archivo CSV con el registro de ejecución).
- **Formato del CSV:** El CSV de entrada utiliza convención europea (punto y coma como separador, coma como decimal).
- **Sistemas de referencia:** Los códigos EPSG de origen y destino para la transformación de coordenadas.
- **Static API:** La URL base y los parámetros que se enviarán en cada petición.
- **Red:** El tiempo máximo de espera por petición, la pausa entre descargas por si existe control de tasa de peticiones (rate-limiting preventivo), y la política de reintentos ante errores transitorios.
- **Recorte posterior:** El tamaño de la imagen descargada y el tamaño del recorte uniforme aplicado.
- **Control de ejecución:** Parámetros opcionales para procesar solo un subconjunto de estructuras durante pruebas.


In [ ]:
# Todos los parametros del pipeline se centralizan en este diccionario 
# Para poder ajustar cualquier valor sin buscar a traves del codigo, se agruparon por bloques tematicos: 
CONFIG = {
    # --- Rutas de entrada/salida ---
    # El CSV entra desde la raiz del proyecto
    # Las imagenes se guardan en una unica carpeta organizada por clase
    "csv_entrada": "Dataset_Maestro_600_CSV_Static_API.csv",
    "carpeta_dataset": "dataset_600_static_api",
    "log_csv": "log_descarga_static_api.csv",

    # --- Formato del CSV (convencion europea) ---
    "csv_separador": ";",
    "csv_decimal": ",",

    # --- Sistemas de referencia ---
    # El CSV usa Web Mercator (metros) tras exportarlo desde ArGIS Pro (capa de puntos)
    # La Static API en cambio espera lat/long en WGS 84 (grados)
    "crs_origen": "EPSG:3857",   # Web Mercator (proyeccion planar)
    "crs_destino": "EPSG:4326",  # WGS 84 (geografica)

    # --- Parametros de la Static API ---
    # size=640x640 + scale=2 --> Tamano y resolucion espacial maxima permitida por la cuota gratuita
    # El style desactiva todas las etiquetas del modo satelital
    "static_api_url": "https://maps.googleapis.com/maps/api/staticmap",
    "size": "640x640",
    "scale": 2,
    "maptype": "satellite",
    "style_labels_off": "feature:all|element:labels|visibility:off",

    # --- Red ---
    # Configuraciones de tiempo de espera de descarga conservadoras 
    "timeout_s": 30,
    "pausa_entre_descargas_s": 0.15,
    "max_reintentos_por_estructura": 3,
    "espera_reintento_base_s": 5,

    # --- Recorte posterior a la descarga ---
    # Elimina franja de atribucion inferior de las imagenes para efectos del entrenamiento
    # El tamano de recorte es multiplo de 32 (preferencia al entrenar YOLOv8)
    "tamano_original": 1280,
    "tamano_recorte": 1184,

    # --- Control de ejecucion ---
    # Estos parametros sirven para pruebas:
    # Procesar solo una clase o limitar la descarga a las primeras N estructuras
    # En None procesa todo el CSV
    "limite_estructuras": None,
    "filtro_clase": None,
    "intervalo_reporte_progreso": 20,
}


### **Sobre los parámetros de la Static API**

Los parámetros que se envían con cada petición determinan el contenido y las propiedades técnicas de la imagen retornada por Google. La combinación elegida para cada estructura del CSV de entrada es el resultado de una calibración empírica en función del nivel de zoom y de las coordenadas específicas de su centroide. 

- **`size=640x640`** — dimensión lógica solicitada. La API interpreta este valor como la extensión visible del mapa antes de aplicar el factor de escala.
- **`scale=2`** — factor multiplicador de la resolución en píxeles. Combinado con el `size` anterior, produce una imagen final de 1280 × 1280 píxeles y pide al servidor descargarla desde el mosaico satelital de mayor detalle disponible al respectivo zoom. Es el mayor factor permitido por la cuota gratuita.
- **`maptype=satellite`** — selecciona la capa de imagen satelital en lugar de las capas cartográficas (`roadmap`, `terrain`, `hybrid`).
- **`style=feature:all|element:labels|visibility:off`** — instrucción de estilización que desactiva la superposición de rótulos toponímicos y viales. Es un parámetro documentado y respetado por la Static API que produce imágenes limpias sin postprocesamiento adicional.

Los parámetros **`center`** y **`zoom`** se completan dinámicamente por cada estructura con sus coordenadas geográficas del centroide en EPSG:4326 (decimales) y su nivel de zoom.

<h2 style="font-weight: bold; border-bottom: 3px solid #37474f; padding-bottom: 8px;">4. Transformación de coordenadas</h2>

El CSV de entrada almacena la localización de cada estructura en el sistema de coordenadas Web Mercator (EPSG:3857), en el que las coordenadas se expresan en metros desde el meridiano de Greenwich y el ecuador. La Static API, en cambio, requiere coordenadas geográficas en grados decimales (EPSG:4326).

La transformación entre ambos sistemas se realiza con `pyproj`, que implementa las fórmulas cartográficas oficiales. La función utiliza un objeto `Transformer` reutilizable: se construye una única vez y se conserva en una variable global de módulo (`_transformer_cache`) para evitar el coste de inicialización en cada llamada, ya que este objeto es relativamente pesado de crear pero muy rápido de usar.

In [ ]:
_transformer_cache = None


def obtener_transformer():
    """Devuelve un Transformer configurado para EPSG:3857 -> EPSG:4326.

    Implementa un patron singleton perezoso: la primera invocacion
    construye el Transformer y las siguientes reutilizan la instancia
    almacenada en la variable global _transformer_cache.

    El argumento always_xy=True garantiza que las coordenadas se
    interpreten en orden (X, Y) tanto en la entrada como en la salida,
    lo que corresponde a (longitud, latitud) en EPSG:4326.
    """
    global _transformer_cache
    if _transformer_cache is None:
        _transformer_cache = Transformer.from_crs(
            CONFIG["crs_origen"],
            CONFIG["crs_destino"],
            always_xy=True,
        )
    return _transformer_cache


def transformar_coordenada(x_3857, y_3857):
    """Convierte coordenadas Web Mercator a geograficas.

    Parametros
    ----------
    x_3857, y_3857 : float
        Coordenadas en EPSG:3857, expresadas en metros.

    Retorna
    -------
    tuple (lon, lat)
        Longitud y latitud en grados decimales.

    Excepciones
    -----------
    ValueError si el resultado cae fuera del rango geografico valido.
    Este chequeo defiende contra corrupciones silenciosas del CSV.
    """
    transformer = obtener_transformer()
    lon, lat = transformer.transform(x_3857, y_3857)
    if not (-180.0 <= lon <= 180.0):
        raise ValueError(f"Longitud {lon:.4f} fuera del rango [-180, 180].")
    if not (-90.0 <= lat <= 90.0):
        raise ValueError(f"Latitud {lat:.4f} fuera del rango [-90, 90].")
    return lon, lat

<h2 style="font-weight: bold; border-bottom: 3px solid #37474f; padding-bottom: 8px;">5. Construcción de la URL de descarga</h2>

Cada petición a la Static API se codifica como una URL con parámetros en formato `clave=valor` unidos por el símbolo `&`. La función `construir_url_static_api` compone esta cadena a partir del centro geográfico y el zoom específicos de cada estructura, junto con los parámetros fijos definidos en `CONFIG`.

El orden de los parámetros no afecta funcionalmente al resultado, pero se mantiene consistente para facilitar la inspección visual de las URLs durante depuración. 

In [ ]:
def construir_url_static_api(lat, lon, zoom):
    """Construye la URL de descarga para una estructura especifica.

    Parametros
    ----------
    lat, lon : float
        Coordenadas geograficas del centro de la imagen.
    zoom : int
        Nivel de zoom (11-21).

    Retorna
    -------
    str
        URL completa lista para ser enviada por HTTP GET.
    """
    params = {
        "center": f"{lat},{lon}",
        "zoom": zoom,
        "size": CONFIG["size"],
        "scale": CONFIG["scale"],
        "maptype": CONFIG["maptype"],
        "style": CONFIG["style_labels_off"],
        "key": API_KEY,
    }
    query = "&".join([f"{k}={v}" for k, v in params.items()])
    return f"{CONFIG['static_api_url']}?{query}"

<h2 style="font-weight: bold; border-bottom: 3px solid #37474f; padding-bottom: 8px;">6. Descarga con manejo robusto de errores</h2>

La comunicación con servicios externos por HTTP puede fallar por múltiples razones: caídas transitorias de red, saturación temporal del servidor, límites de tasa alcanzados, o problemas de autenticación. La función `descargar_imagen_con_reintentos` implementa una política de reintentos diferenciada según el tipo de error.

### **Política de reintentos**

La función clasifica cada respuesta o excepción en tres categorías:

- **Éxito (HTTP 200 con contenido de imagen)** — se retorna el contenido binario para procesamiento inmediato.
- **Error transitorio (HTTP 429 o 5xx, timeouts)** — se aplica una espera exponencial (5 s, 10 s, 20 s) antes de reintentar. Este esquema, conocido como *exponential backoff*, permite que el servicio se recupere sin saturarlo con reintentos inmediatos.
- **Error permanente (HTTP 400, 401, 403)** — se retorna inmediatamente como error definitivo. Reintentar una petición mal autenticada o mal formada no la hará exitosa.

El máximo de reintentos y la espera base se controlan desde `CONFIG` para facilitar ajustes sin modificar la función.

In [ ]:
def descargar_imagen_con_reintentos(url, timeout=None):
    """Descarga la imagen desde la URL con manejo diferenciado de errores.

    Aplica reintentos con backoff exponencial ante errores transitorios
    (rate limits, timeouts, errores 5xx) y aborta inmediatamente ante
    errores permanentes (4xx no reintentables).

    Retorna
    -------
    dict con las claves:
      - status : 'OK' o 'ERROR'
      - bytes : contenido binario si OK, None si ERROR
      - mensaje : descripcion textual del error si ERROR
      - http_status : ultimo codigo HTTP observado, o None
    """
    if timeout is None:
        timeout = CONFIG["timeout_s"]

    ultimo_error = None
    ultimo_http_status = None

    for intento in range(1, CONFIG["max_reintentos_por_estructura"] + 1):
        try:
            response = requests.get(url, timeout=timeout)
            ultimo_http_status = response.status_code

            # Caso exito
            if response.status_code == 200:
                content_type = response.headers.get('Content-Type', '')
                if 'image' not in content_type.lower():
                    return {
                        'status': 'ERROR',
                        'bytes': None,
                        'mensaje': f'Respuesta no es imagen: {content_type}. '
                                   f'Cuerpo: {response.text[:200]}',
                        'http_status': response.status_code,
                    }
                return {
                    'status': 'OK',
                    'bytes': response.content,
                    'mensaje': '',
                    'http_status': 200,
                }

            # Rate limit: backoff exponencial
            if response.status_code == 429:
                espera = CONFIG["espera_reintento_base_s"] * (2 ** (intento - 1))
                print(f"       [WARN] Rate limit HTTP 429. Esperando {espera}s...")
                time.sleep(espera)
                ultimo_error = f'HTTP 429 tras {intento} intentos'
                continue

            # Errores 4xx no reintentables: abortar
            if 400 <= response.status_code < 500 and response.status_code != 429:
                return {
                    'status': 'ERROR',
                    'bytes': None,
                    'mensaje': f'HTTP {response.status_code} (no reintentable): '
                               f'{response.text[:200]}',
                    'http_status': response.status_code,
                }

            # Errores 5xx: reintentar
            espera = CONFIG["espera_reintento_base_s"] * (2 ** (intento - 1))
            print(f"       [WARN] HTTP {response.status_code}. Esperando {espera}s...")
            time.sleep(espera)
            ultimo_error = f'HTTP {response.status_code} tras {intento} intentos'

        except requests.exceptions.Timeout:
            espera = CONFIG["espera_reintento_base_s"] * (2 ** (intento - 1))
            print(f"       [WARN] Timeout. Esperando {espera}s y reintentando...")
            time.sleep(espera)
            ultimo_error = f'Timeout tras {intento} intentos'

        except requests.exceptions.RequestException as e:
            return {
                'status': 'ERROR',
                'bytes': None,
                'mensaje': f'{type(e).__name__}: {e}',
                'http_status': None,
            }

    return {
        'status': 'ERROR',
        'bytes': None,
        'mensaje': ultimo_error or 'Fallo tras todos los reintentos',
        'http_status': ultimo_http_status,
    }

<h2 style="font-weight: bold; border-bottom: 3px solid #37474f; padding-bottom: 8px;">7. Recorte geométrico centrado</h2> 

Cada imagen descargada tiene 1280×1280 píxeles y se destina a producir un recorte centrado de 1184×1184 píxeles. La función `calcular_recorte_centrado` produce las cuatro coordenadas (left, top, right, bottom) que `Pillow` necesita para el método `crop`.

El recorte es siempre simétrico: se descarta la misma cantidad de píxeles por cada borde, lo que garantiza que el centro geométrico de la imagen recortada coincida con el centro de la imagen original, y por tanto con el centro geográfico de la consulta a la Static API. Esta propiedad es importante porque significa que la estructura de interés, situada por construcción en el centro geográfico, sigue estando en el centro visual del recorte final.


In [ ]:
def calcular_recorte_centrado(tamano_original, tamano_recorte):
    """Calcula las coordenadas de un recorte cuadrado centrado.

    Retorna una tupla (left, top, right, bottom) compatible con
    PIL.Image.crop(), que define un area cuadrada de tamano_recorte
    pixeles centrada dentro de una imagen de tamano_original.
    """
    margen = (tamano_original - tamano_recorte) // 2
    return (margen, margen,
            tamano_original - margen,
            tamano_original - margen)


def construir_ruta_salida(carpeta_raiz, clase, id_estructura):
    """Construye la ruta destino siguiendo la jerarquia por clase.

    Convencion: <carpeta_raiz>/<Clase_sanitizada>/<id>.png
    Los espacios en el nombre de la clase se reemplazan por guiones
    bajos para asegurar compatibilidad con todos los sistemas de
    archivos y facilitar el trabajo posterior desde la linea de
    comandos.
    """
    clase_sanitizada = clase.replace(" ", "_")
    return Path(carpeta_raiz) / clase_sanitizada / f"{id_estructura}.png"


<h2 style="font-weight: bold; border-bottom: 3px solid #37474f; padding-bottom: 8px;">8. Procesamiento de una estructura individual</h2>

La función `procesar_estructura` orquesta el flujo completo para una fila del CSV: transforma coordenadas, construye la URL, descarga la imagen, genera el recorte centrado y lo guarda en disco. Retorna un diccionario con toda la información necesaria para registrar el resultado en el log. 

Una decisión de diseño es que la imagen original 1280 × 1280 no se persiste en el disco. Se mantiene en memoria durante el tiempo suficiente para producir el recorte, y luego el intérprete de Python libera la memoria automáticamente al pasar a la siguiente estructura. Este enfoque evita duplicar la ocupación en disco sin perder información útil, ya que el recorte final conserva íntegramente la región de interés en torno al centro geográfico de la consulta.


In [ ]:
def procesar_estructura(fila):
    """Procesa una fila del CSV: descarga + genera 1 recorte + guarda.

    Recibe una fila del DataFrame (objeto pandas.Series) con las
    columnas id, clase, X_3857, Y_3857 y zoom. Retorna un diccionario
    con el resultado detallado del procesamiento.
    """
    id_est = int(fila['id'])
    clase = fila['clase']
    zoom = int(fila['zoom'])

    resultado = {
        'id': id_est,
        'clase': clase,
        'zoom': zoom,
        'lon': None,
        'lat': None,
        'url_sin_key': '',
        'ruta': '',
        'tamano_recorte': None,
        'bytes_descargados': 0,
        'estado': 'ERROR',
        'mensaje': '',
    }

    # Paso 1: transformar coordenadas al sistema geografico
    lon, lat = transformar_coordenada(fila['X_3857'], fila['Y_3857'])
    resultado['lon'] = lon
    resultado['lat'] = lat

    # Paso 2: construir URL y guardar version sin API key para el log
    url = construir_url_static_api(lat, lon, zoom)
    resultado['url_sin_key'] = url.split('&key=')[0]

    # Paso 3: descargar con manejo robusto de errores
    respuesta = descargar_imagen_con_reintentos(url)
    if respuesta['status'] != 'OK':
        resultado['mensaje'] = respuesta['mensaje']
        return resultado

    resultado['bytes_descargados'] = len(respuesta['bytes'])

    # Paso 4: cargar imagen en memoria desde los bytes descargados
    imagen_original = Image.open(BytesIO(respuesta['bytes']))
    ancho, alto = imagen_original.size

    # Validacion defensiva del tamano recibido
    if ancho != CONFIG["tamano_original"] or alto != CONFIG["tamano_original"]:
        resultado['mensaje'] = (
            f'Tamano inesperado de imagen descargada: {ancho}x{alto} '
            f'(esperado {CONFIG["tamano_original"]}x{CONFIG["tamano_original"]})'
        )
        return resultado

    # Paso 5: recorte centrado uniforme
    tamano = CONFIG["tamano_recorte"]
    coords = calcular_recorte_centrado(CONFIG["tamano_original"], tamano)
    imagen_recortada = imagen_original.crop(coords)
    ruta = construir_ruta_salida(CONFIG["carpeta_dataset"], clase, id_est)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    imagen_recortada.save(ruta, "PNG")
    resultado['ruta'] = str(ruta)
    resultado['tamano_recorte'] = tamano

    resultado['estado'] = 'OK'
    return resultado


<h2 style="font-weight: bold; border-bottom: 3px solid #37474f; padding-bottom: 8px;">9. Registro y reanudación</h2> 

El log CSV cumple dos funciones simultáneas: sirve como registro trazable de cada descarga (útil para auditoría y análisis posteriores) y como base para la reanudación automática si el flujo se interrumpe.

La reanudación funciona así: al iniciar el procesamiento, se lee el log existente y se identifican los IDs marcados como `OK`. Estas estructuras se omiten en el bucle principal, y el flujo solo procesa las que están pendientes o han fallado en ejecuciones anteriores. Esto permite recuperar la ejecución tras cualquier interrupción sin perder progreso ni consumir cuota adicional por descargas repetidas.

El log se escribe fila a fila, en modo *append*, con un flush implícito tras cada operación. De este modo, incluso una interrupción abrupta (cierre forzado del proceso, corte de energía) preserva el estado hasta la última descarga completada.

In [ ]:
def cargar_log_existente(ruta_log):
    """Lee el log de descargas previas y retorna el conjunto de IDs OK.

    Si el archivo no existe (primera ejecucion), retorna un conjunto
    vacio. Si existe pero no se puede leer, emite una advertencia y
    continua como si fuera una primera ejecucion (comportamiento
    conservador que evita perder progreso ante corrupciones parciales).
    """
    if not ruta_log.exists():
        return set()
    try:
        df_log = pd.read_csv(ruta_log)
        ya_descargados = set(df_log[df_log["estado"] == "OK"]["id"].astype(int))
        return ya_descargados
    except Exception as e:
        print(f"  [ADVERTENCIA] No se pudo leer el log existente: {e}")
        return set()


def registrar_en_log(ruta_log, fila_log):
    """Anade una fila al log CSV, creando cabecera si es la primera."""
    columnas = [
        "timestamp", "id", "clase", "zoom", "lon", "lat",
        "url_sin_key", "ruta", "tamano_recorte",
        "bytes_descargados", "estado", "mensaje",
    ]
    archivo_existe = ruta_log.exists()
    with open(ruta_log, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=columnas)
        if not archivo_existe:
            writer.writeheader()
        writer.writerow(fila_log)


def imprimir_progreso(actual, total, exitosos, fallidos, saltados, tiempo_inicio):
    """Muestra una linea de progreso con porcentaje y estimacion de fin.

    Se llama periodicamente durante el bucle principal para dar
    feedback al usuario sobre el avance del procesamiento.
    """
    elapsed = time.time() - tiempo_inicio
    if actual > 0 and elapsed > 0.001:
        velocidad = actual / elapsed
        restantes = total - actual
        eta_segundos = restantes / velocidad if velocidad > 0 else 0
        eta_min = eta_segundos / 60
    else:
        eta_min = 0
    porcentaje = (actual / total * 100) if total > 0 else 0
    print(f"  [{actual:>4d}/{total:>4d}] {porcentaje:5.1f}%  "
          f"OK={exitosos:>4d}  ERR={fallidos:>3d}  SKIP={saltados:>4d}  "
          f"elapsed={elapsed/60:5.1f}min  ETA={eta_min:5.1f}min")


<h2 style="font-weight: bold; border-bottom: 3px solid #37474f; padding-bottom: 8px;">10. Bucle principal de procesamiento</h2>

La función `procesar_dataset` implementa el bucle que recorre todas las estructuras del CSV y las procesa una por una. Antes de iniciar, aplica los filtros opcionales de clase y límite (útiles durante pruebas) y consulta el log existente para identificar las estructuras ya descargadas.

Para cada estructura, verifica primero si ya fue procesada exitosamente (por su presencia en el log o por la existencia del archivo PNG correspondiente en disco). Si es así, la salta silenciosamente. En caso contrario, invoca `procesar_estructura` y registra el resultado en el log con marca de tiempo.

Entre descargas se aplica una pausa configurable (`pausa_entre_descargas_s`) que actúa como *rate limiting* preventivo. Esta pausa mantiene el flujo por debajo de los límites de la API sin necesidad de reactivos, y suaviza la carga sobre el servidor de Google.


In [ ]:
def procesar_dataset(df, ruta_log):
    """Recorre el DataFrame y procesa cada estructura secuencialmente.

    Aplica los filtros opcionales de CONFIG (clase, limite), gestiona
    la reanudacion desde log previo, y produce un resumen final con
    contadores de exitos, errores y saltos.
    """
    filtro = CONFIG["filtro_clase"]
    if filtro is not None:
        df = df[df["clase"] == filtro].copy()
        print(f"  FILTRO ACTIVO: solo se procesan estructuras de clase '{filtro}' "
              f"({len(df)} encontradas).")
    limite = CONFIG["limite_estructuras"]
    if limite is not None and limite > 0:
        df = df.head(limite)
        print(f"  LIMITE ACTIVO: solo las primeras {limite} estructuras.")

    total = len(df)
    if total == 0:
        print(f"  [ADVERTENCIA] No hay estructuras para procesar tras filtros.")
        return {"exitosos": 0, "fallidos": 0, "saltados": 0, "total": 0}

    ya_descargados = cargar_log_existente(ruta_log)
    if ya_descargados:
        print(f"  Reanudacion: {len(ya_descargados)} estructuras ya descargadas previamente.")

    exitosos = 0
    fallidos = 0
    saltados = 0
    tiempo_inicio = time.time()

    print(f"\n  Iniciando procesamiento de {total} estructuras...")
    print(f"  Cada descarga genera 1 imagen recortada centrada.\n")

    for indice, (_, fila) in enumerate(df.iterrows(), start=1):
        id_estructura = int(fila["id"])
        clase = fila["clase"]

        # Verificacion de reanudacion: PNG debe existir en la carpeta destino
        ruta = construir_ruta_salida(CONFIG["carpeta_dataset"], clase, id_estructura)
        if id_estructura in ya_descargados or ruta.exists():
            saltados += 1
            if indice % CONFIG["intervalo_reporte_progreso"] == 0:
                imprimir_progreso(indice, total, exitosos, fallidos,
                                  saltados, tiempo_inicio)
            continue

        timestamp = datetime.now().isoformat(timespec="seconds")
        try:
            resultado = procesar_estructura(fila)

            if resultado['estado'] == 'OK':
                exitosos += 1
                registrar_en_log(ruta_log, {
                    "timestamp": timestamp,
                    "id": resultado['id'],
                    "clase": resultado['clase'],
                    "zoom": resultado['zoom'],
                    "lon": f"{resultado['lon']:.6f}",
                    "lat": f"{resultado['lat']:.6f}",
                    "url_sin_key": resultado['url_sin_key'],
                    "ruta": resultado['ruta'],
                    "tamano_recorte": resultado['tamano_recorte'],
                    "bytes_descargados": resultado['bytes_descargados'],
                    "estado": "OK",
                    "mensaje": "",
                })
                kb = resultado['bytes_descargados'] / 1024
                print(f"  [OK id={id_estructura:>3d} clase={clase:<15s}] "
                      f"zoom={resultado['zoom']:>2d}  {kb:>5.1f}KB  "
                      f"recorte={resultado['tamano_recorte']}px")
            else:
                fallidos += 1
                print(f"\n  [ERROR id={id_estructura} clase={clase}] "
                      f"{resultado['mensaje'][:200]}\n")
                registrar_en_log(ruta_log, {
                    "timestamp": timestamp,
                    "id": id_estructura,
                    "clase": clase,
                    "zoom": int(fila['zoom']),
                    "lon": f"{resultado['lon']:.6f}" if resultado['lon'] else "",
                    "lat": f"{resultado['lat']:.6f}" if resultado['lat'] else "",
                    "url_sin_key": resultado['url_sin_key'],
                    "ruta": "",
                    "tamano_recorte": "",
                    "bytes_descargados": resultado['bytes_descargados'],
                    "estado": "ERROR",
                    "mensaje": resultado['mensaje'][:500],
                })

        except Exception as e:
            fallidos += 1
            mensaje_error = f"{type(e).__name__}: {str(e)[:200]}"
            print(f"\n  [EXCEPCION id={id_estructura} clase={clase}] {mensaje_error}\n")
            registrar_en_log(ruta_log, {
                "timestamp": timestamp,
                "id": id_estructura,
                "clase": clase,
                "zoom": int(fila.get('zoom', 0)),
                "lon": "",
                "lat": "",
                "url_sin_key": "",
                "ruta": "",
                "tamano_recorte": "",
                "bytes_descargados": 0,
                "estado": "EXCEPCION",
                "mensaje": mensaje_error,
            })

        # Rate limiting preventivo entre descargas consecutivas
        time.sleep(CONFIG["pausa_entre_descargas_s"])

        if indice % CONFIG["intervalo_reporte_progreso"] == 0:
            imprimir_progreso(indice, total, exitosos, fallidos,
                              saltados, tiempo_inicio)

    imprimir_progreso(total, total, exitosos, fallidos, saltados, tiempo_inicio)

    return {
        "exitosos": exitosos,
        "fallidos": fallidos,
        "saltados": saltados,
        "total": total,
    }


<h2 style="font-weight: bold; border-bottom: 3px solid #37474f; padding-bottom: 8px;">11. Validaciones previas y ejecución del pipeline</h2> 
Antes de invocar el bucle principal, se realizan tres verificaciones que evitan errores tardíos y consumo innecesario de cuota:

1. **Presencia de la API key** — la ejecución se detiene si aún contiene el placeholder.
2. **Existencia del CSV** — se comprueba la ruta indicada en `CONFIG`.
3. **Integridad del CSV** — se verifican las columnas requeridas y se emite una descripción de la distribución por clase para inspección visual antes de iniciar.

In [ ]:
# Validacion 1: API key configurada
if API_KEY == "TU_API_KEY_AQUI":
    print("[ERROR] Debes editar la celda 2 y reemplazar 'TU_API_KEY_AQUI'")
    print("        por tu API key real antes de ejecutar este flujo.")
    raise SystemExit("API key no configurada")

# Validacion 2: CSV existe
csv_path = Path(CONFIG["csv_entrada"])
if not csv_path.exists():
    print(f"[ERROR] No se encontro el CSV: {csv_path.absolute()}")
    raise SystemExit("CSV faltante")

# Lectura y validacion 3: columnas requeridas
df = pd.read_csv(csv_path, sep=CONFIG["csv_separador"],
                 decimal=CONFIG["csv_decimal"])
print(f"[OK] CSV leido: {len(df)} filas")
print(f"     Columnas: {list(df.columns)}")

columnas_requeridas = {'id', 'clase', 'X_3857', 'Y_3857', 'zoom'}
columnas_faltantes = columnas_requeridas - set(df.columns)
if columnas_faltantes:
    print(f"[ERROR] El CSV no tiene las columnas requeridas: {columnas_faltantes}")
    raise SystemExit("CSV con columnas faltantes")

print(f"\nDistribucion por clase:")
for clase, n in df['clase'].value_counts().sort_index().items():
    print(f"  {clase:20s}: {n:3d}")

print(f"\nParametros de descarga:")
print(f"  Tamano de recorte:  {CONFIG['tamano_recorte']}x{CONFIG['tamano_recorte']} px")
print(f"  Carpeta de salida:  {Path(CONFIG['carpeta_dataset']).absolute()}")
print(f"  Log:                {Path(CONFIG['log_csv']).absolute()}")


### **Ejecución del bucle principal**

La celda siguiente inicia el procesamiento del dataset completo. En condiciones normales de red, las 600 estructuras se procesan en aproximadamente 17 minutos. Durante la ejecución, se muestra una línea por cada descarga exitosa con el ID, la clase, el zoom, el tamaño del archivo descargado y el tamaño del recorte aplicado. Cada veinte estructuras se emite un resumen agregado con el porcentaje de progreso y el tiempo estimado hasta la finalización.

La ejecución puede interrumpirse en cualquier momento (por ejemplo con `Kernel > Interrupt` en Jupyter). Al reejecutar esta celda posteriormente, el flujo reanuda desde donde se detuvo.


In [ ]:
ruta_log = Path(CONFIG["log_csv"])
print("Iniciando descargas...")
resumen = procesar_dataset(df, ruta_log)

<h2 style="font-weight: bold; border-bottom: 3px solid #37474f; padding-bottom: 8px;">12. Resumen final</h2>

Al finalizar el procesamiento, esta celda imprime un resumen consolidado con los contadores absolutos y las rutas de los artefactos producidos. Si hay errores pendientes, informa que una nueva ejecución los reintentará automáticamente sin repetir las descargas exitosas.

In [ ]:
print("=" * 72)
print(" RESUMEN FINAL")
print("=" * 72)
print(f"  Total procesado:    {resumen['total']}")
print(f"  Descargados (OK):   {resumen['exitosos']}")
print(f"  Saltados (ya OK):   {resumen['saltados']}")
print(f"  Errores:            {resumen['fallidos']}")
print(f"\n  Imagenes en:  {Path(CONFIG['carpeta_dataset']).absolute()}")
print(f"  Log en:       {ruta_log.absolute()}")
if resumen['fallidos'] > 0:
    print(f"\n  Hay {resumen['fallidos']} errores. Vuelve a ejecutar la celda del bucle")
    print(f"  principal para reintentarlos (la reanudacion saltara los exitosos).")
print("=" * 72)


<h2 style="font-weight: bold; border-bottom: 3px solid #37474f; padding-bottom: 8px;">13. Verificación de coherencia</h2> 

Como paso final, es recomendable verificar que el número de archivos en disco coincide con el número de registros exitosos en el log. Cualquier discrepancia entre ambos indica una corrupción o interrupción anómala que merece inspección manual.

In [ ]:
pngs = list(Path(CONFIG["carpeta_dataset"]).rglob("*.png"))
log_df = pd.read_csv(ruta_log)
ids_log = set(log_df[log_df["estado"] == "OK"]["id"].astype(int))
ids_disco = set(int(p.stem) for p in pngs)

print(f"PNGs en carpeta de salida: {len(pngs)}")
print(f"Registros OK en log:       {len(ids_log)}")

discrepancias = ids_disco ^ ids_log
print(f"Discrepancias disco vs log: {'ninguna' if not discrepancias else discrepancias}")
